# Embeddings: e5 (dense) and BM25 (sparse)

Two ways to turn text into vectors, and why both are used.

See the [README](README.md) for the why and the how.

In [ ]:
# Run from anywhere inside the repository: find its root so `marginalia` and the test helpers import,
# and read .env (the same settings the services use).
import os, sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / 'pyproject.toml').exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print(ROOT)

## Dense: e5
The first run downloads `intfloat/e5-base-v2` (ONNX, ~440 MB) into the fastembed cache.

In [ ]:
import numpy as np
from marginalia.index.embeddings import E5Embedder
e5 = E5Embedder()
print('dimensions:', e5.dim)
passages = ['Raskolnikov killed the old pawnbroker with an axe.',
            'Sonia read the story of Lazarus to Raskolnikov.',
            'The weather in Petersburg was hot and stifling.']
P = np.array(e5.embed_passages(passages))
q = np.array(e5.embed_query('Who did Raskolnikov murder?'))
for text, score in sorted(zip(passages, P @ q), key=lambda x: -x[1]):
    print(f'{score:.3f}  {text}')

## Why the "query:" / "passage:" prefixes matter
e5 was trained with them; the embedder adds them, so callers never do.

In [ ]:
raw = np.array([v.tolist() for v in e5._model.embed(['Who did Raskolnikov murder?'])])[0]
print('with prefix  :', np.round(P @ q, 3))
print('without      :', np.round(P @ raw, 3))

## Sparse: BM25
Exact words count: names, rare terms. Qdrant applies the IDF part at query time.

In [ ]:
from marginalia.index.embeddings import Bm25Encoder, fold_diacritics
bm25 = Bm25Encoder()
v = bm25.embed_query('Svidrigailov and Raskolnikov')
print(len(v.indices), 'terms', v.indices[:5], v.values[:5])
print(fold_diacritics('Svidrigaïlov'), '<- diacritics folded, so a typed name matches the book')

## Offline stand-ins
The tests use deterministic bag-of-words vectors instead of models.

In [ ]:
from tests.conftest import FakeDense
fd = FakeDense()
print(len(fd.embed_query('the axe')), np.round(np.array(fd.embed_query('the axe')) @ np.array(fd.embed_passages(['the axe fell'])[0]), 3))